# RevRAG — Kaggle Experiment Notebook

**GPU:** T4 (16 GB VRAM) — Settings → Accelerator → GPU T4 x1  
**Internet:** must be ON — Settings → Internet → On  
**Session time:** up to 9 hours.

### Primary benchmark datasets (all real, no synthetic data for main claims)
| Dataset | Source | What it tests |
|---|---|---|
| LegalBench-RAG | zeroentropy-ai/legalbenchrag (GitHub) | Character-level retrieval over real contracts |
| ContractNLI | koreeda/ContractNLI (HuggingFace) | Retrieval over real NDAs; Contradiction subset = cross-page test |
| CUAD Amendments | theatticusproject/cuad (HuggingFace) | Real expert-annotated amendment pairs from 510 contracts |
| MultiHop-RAG | yixuantt/MultiHopRAG (HuggingFace) | Peer-reviewed multi-hop retrieval, non-legal domain |
| LoCoV1 | hazyresearch/LoCoV1 (HuggingFace) | Long-context retrieval up to 32K tokens |
| TREC DL | MS MARCO (requires manual download) | Standard IR benchmark — run on cluster |

### Supplementary (diagnostic only)
| Dataset | Role |
|---|---|
| Synthetic cross-page benchmark | Ablation / error analysis only — not used for main claims |

### ⚠️ Mamba-2 note
`mamba-ssm` requires CUDA 11.8+ and ~8 min to compile from source.  
If it fails, the code falls back to a GRU stub — pipeline runs but encoder numbers are meaningless.

## Cell 1 — Verify GPU

In [ ]:
import subprocess, sys
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
import torch
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 2 — Install dependencies (~5–10 min)

In [ ]:
%%bash
set -e

pip install -q \
    transformers==4.44.0 \
    datasets \
    faiss-gpu \
    rank-bm25 \
    sentence-transformers \
    FlagEmbedding \
    pytrec-eval-terrier \
    tqdm pyyaml numpy scipy matplotlib openai

echo '✓ Core packages installed'

# mamba-ssm — CUDA required
pip install -q causal-conv1d 2>/dev/null || true
pip install -q mamba-ssm 2>/dev/null || (
    echo 'pip wheel failed — building from source (~8 min)'
    pip install -q ninja packaging
    pip install -q git+https://github.com/state-spaces/mamba.git
)
python -c "import mamba_ssm; print('✓ mamba-ssm OK')" 2>/dev/null || \
    echo '⚠ mamba-ssm unavailable — GRU stub will be used'

echo 'Done.'

## Cell 3 — Upload or clone RevRAG code

**Option A:** Push `F:\MambaRAG\revrag\` to a private GitHub repo → set `USE_GITHUB=True` and fill `GITHUB_REPO`.  
**Option B:** Zip `F:\MambaRAG\revrag\` → upload via Kaggle Add Data panel → unzip here.

In [ ]:
import os, sys, zipfile

GITHUB_REPO = 'https://github.com/YOUR_USERNAME/revrag.git'  # <-- change
USE_GITHUB  = False
WORK_DIR    = '/kaggle/working/revrag'

if USE_GITHUB:
    os.system(f'git clone {GITHUB_REPO} {WORK_DIR}')
else:
    zip_path = '/kaggle/input/revrag/revrag.zip'
    if os.path.exists(zip_path):
        with zipfile.ZipFile(zip_path) as z:
            z.extractall('/kaggle/working/')
        print(f'Extracted → {WORK_DIR}')
    else:
        print('⚠ Upload revrag.zip via the Data panel first.')

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)
print('CWD:', os.getcwd())
print(sorted(os.listdir('.')))

## Cell 4 — Download all benchmark datasets

All datasets load via HuggingFace `datasets` or a git clone.  
Estimated download time: ~5 min total (datasets are cached after first run).

| Dataset | Size | Time |
|---|---|---|
| LegalBench-RAG | ~50 MB | ~1 min |
| ContractNLI | ~30 MB | ~30 sec |
| CUAD | ~300 MB | ~2 min |
| MultiHop-RAG | ~200 MB (500 queries subset) | ~1 min |
| LoCoV1 | ~500 MB | ~2 min |

In [ ]:
%%bash
cd /kaggle/working/revrag
mkdir -p data/raw

# 1. LegalBench-RAG
if [ ! -d data/raw/legalbench_rag ]; then
    git clone --depth=1 https://github.com/zeroentropy-ai/legalbenchrag \
        data/raw/legalbench_rag
    echo '✓ LegalBench-RAG'
else
    echo '✓ LegalBench-RAG (cached)'
fi

# 2–5. HuggingFace datasets (ContractNLI, CUAD, MultiHopRAG, LoCoV1)
# These are streamed/cached automatically when load_dataset() is called.
# Pre-download them now so eval cells don't stall mid-run.
python - <<'EOF'
from datasets import load_dataset
print('Downloading ContractNLI...')
load_dataset('koreeda/ContractNLI', cache_dir='data/raw/contractnli')
print('✓ ContractNLI')

print('Downloading CUAD...')
load_dataset('theatticusproject/cuad', cache_dir='data/raw/cuad')
print('✓ CUAD')

print('Downloading MultiHop-RAG (streaming 500 queries)...')
load_dataset('yixuantt/MultiHopRAG', split='MultiHopRAG',
             cache_dir='data/raw/multihop_rag')
print('✓ MultiHop-RAG')

print('Downloading LoCoV1...')
load_dataset('hazyresearch/LoCoV1', split='test',
             cache_dir='data/raw/loco_v1')
print('✓ LoCoV1')
EOF

echo '\n✓ All datasets ready.'

## Cell 5 — Generate supplementary synthetic benchmark

**Role:** diagnostic / ablation only — NOT used for main retrieval claims.  
Used in: λ × τ hyperparameter sweep (Cell 9) and error analysis.
Takes ~3 min.

In [ ]:
import sys, os, json, logging, yaml
os.chdir('/kaggle/working/revrag')
sys.path.insert(0, '.')
logging.basicConfig(level=logging.INFO)

with open('configs/default.yaml') as f:
    cfg = yaml.safe_load(f)

from data.legalbench import LegalBenchRAGDataset
from data.synthetic import generate_synthetic_benchmark

lb = LegalBenchRAGDataset(cfg)
source_docs = lb.get_all_documents()
print(f'Source documents: {len(source_docs)}')

items = generate_synthetic_benchmark(
    source_documents=source_docs,
    output_path='data/synthetic/cross_page_bench.jsonl',
    num_documents=400,
    contradictions_per_doc=5,
    validation_sample=100,
    seed=42,
)
print(f'✓ {len(items)} synthetic items (supplementary only)')

## Cell 6 — Smoke test

In [ ]:
%%bash
cd /kaggle/working/revrag
python tests/smoke_test.py

## Cell 7 — Primary evaluation on real benchmark datasets

All models × all **real** datasets × 3 seeds.

Estimated times on T4:

| Dataset | Per model | Total (6 models × 3 seeds) |
|---|---|---|
| legalbench | ~8 min | ~25 min |
| contractnli | ~10 min | ~30 min |
| cuad_amendments | ~12 min | ~35 min |
| multihop_rag | ~15 min | ~45 min |
| loco | ~20 min | ~60 min |
| **Total** | | **~3.5 hours** |

TREC DL is skipped here (MS MARCO is 20 GB — run on cluster via `eval_all.sbatch`).

In [ ]:
import subprocess, json, sys, os
from pathlib import Path

os.chdir('/kaggle/working/revrag')

MODELS = ['bm25', 'bge', 'bgem3', 'mamba', 'revrag_no_verifier', 'revrag']

# Primary: all real benchmark datasets
REAL_DATASETS = [
    'legalbench',       # char-level retrieval over CUAD/MAUD/ContractNLI/PrivacyQA
    'contractnli',      # expert-annotated NLI spans in real NDAs
    'cuad_amendments',  # real amendment pairs from CUAD expert annotations
    'multihop_rag',     # peer-reviewed multi-hop retrieval (Li et al., 2024)
    'loco',             # long-context retrieval up to 32K tokens
]

SEEDS = [42, 123, 456]
failed = []

for model in MODELS:
    for dataset in REAL_DATASETS:
        for seed in SEEDS:
            tag = f'{model} | {dataset} | seed={seed}'
            print(f'\n{"="*60}\n  {tag}\n{"="*60}')
            cmd = [
                sys.executable, 'evaluate/run_eval.py',
                '--config', 'configs/default.yaml',
                '--model', model,
                '--dataset', dataset,
                '--seeds', str(seed),
                '--device', 'cuda',
            ]
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=1800)
            if result.returncode != 0:
                print(f'❌ FAILED: {tag}')
                print(result.stderr[-800:])
                failed.append(tag)
            else:
                # Print last few lines of stdout (metrics)
                lines = result.stdout.strip().split('\n')
                print('\n'.join(lines[-6:]))

print(f'\n✓ Primary eval complete.')
if failed:
    print(f'⚠ {len(failed)} runs failed: {failed}')

## Cell 7b — Supplementary: synthetic benchmark (diagnostic only)

Run this only for error analysis and verifier ablation, not for main claims.

In [ ]:
import subprocess, sys, os
os.chdir('/kaggle/working/revrag')

MODELS  = ['bm25', 'bge', 'bgem3', 'mamba', 'revrag_no_verifier', 'revrag']
SEEDS   = [42, 123, 456]

for model in MODELS:
    for seed in SEEDS:
        print(f'Synthetic | {model} | seed={seed}')
        cmd = [
            sys.executable, 'evaluate/run_eval.py',
            '--config', 'configs/default.yaml',
            '--model', model,
            '--dataset', 'synthetic',
            '--seeds', str(seed),
            '--device', 'cuda',
        ]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=600)
        lines = result.stdout.strip().split('\n')
        print('\n'.join(lines[-4:]))

print('✓ Supplementary synthetic eval complete.')

## Cell 8 — Aggregate results: mean ± std table across 3 seeds

In [ ]:
import json, numpy as np
from pathlib import Path

results_dir = Path('/kaggle/working/revrag/outputs')
all_runs = list(results_dir.glob('*_seed*.json'))
print(f'Found {len(all_runs)} result files')

from collections import defaultdict
grouped = defaultdict(list)
for f in all_runs:
    with open(f) as fh:
        r = json.load(fh)
    key = (r.get('model', '?'), r.get('dataset', '?'))
    grouped[key].append(r)

summary = {}
for (model, dataset), runs in sorted(grouped.items()):
    agg = {}
    all_keys = set().union(*[r.keys() for r in runs])
    for k in all_keys:
        vals = [r[k] for r in runs if isinstance(r.get(k), (int, float))]
        if vals:
            agg[k] = {'mean': float(np.mean(vals)), 'std': float(np.std(vals))}
    summary[f'{model}/{dataset}'] = agg

# Print compact table
KEY_METRICS = {
    'legalbench':      ('char_recall@5', 'Char-R@5'),
    'contractnli':     ('contradiction_char_recall@5', 'Contra-R@5'),
    'cuad_amendments': ('cra@1', 'CRA@1'),
    'multihop_rag':    ('recall@5', 'R@5'),
    'loco':            ('recall@5', 'R@5'),
    'synthetic':       ('cra@1', 'CRA@1'),
}

print(f'\n{"Model/Dataset":<40} {"Metric":<18} {"Mean":>8} {"±Std":>8}')
print('-' * 80)
for key, metrics in sorted(summary.items()):
    _, dataset = key.split('/', 1)
    metric_key, metric_name = KEY_METRICS.get(dataset, ('char_recall@5', 'Char-R@5'))
    m = metrics.get(metric_key, {})
    mean = m.get('mean', float('nan'))
    std  = m.get('std', float('nan'))
    print(f'{key:<40} {metric_name:<18} {mean:>8.4f} {std:>8.4f}')

with open('/kaggle/working/revrag/outputs/summary_all.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('\n✓ Summary saved → outputs/summary_all.json')

## Cell 9 — Hyperparameter ablation (λ × τ sweep, synthetic diagnostic)

99 configurations on 500 synthetic items. ~30 min on T4.

In [ ]:
%%bash
cd /kaggle/working/revrag
python experiments/ablation_hyperparams.py \
    --config configs/default.yaml \
    --dataset-path data/synthetic/cross_page_bench.jsonl \
    --device cuda \
    --max-items 500 \
    --output-dir outputs/ablation

## Cell 10 — Generation evaluation (optional — needs OpenAI key)

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'  # paste your key — do NOT commit

if os.environ['OPENAI_API_KEY'].startswith('sk-'):
    # Use ContractNLI Contradiction subset for generation eval
    # (real dataset, not synthetic)
    !cd /kaggle/working/revrag && python evaluate/run_generation_eval.py \
        --config configs/default.yaml \
        --dataset-path data/synthetic/cross_page_bench.jsonl \
        --backend openai \
        --model-name gpt-4o-mini \
        --n-queries 200 \
        --compare-all \
        --device cuda \
        --output-dir outputs/generation_eval
else:
    print('No OpenAI key set. Use HF backend:')
    print('  --backend hf --model-name microsoft/phi-3-mini-4k-instruct')

## Cell 11 — Efficiency sweep and plots

In [ ]:
%%bash
cd /kaggle/working/revrag
mkdir -p outputs/figures

python experiments/run_efficiency.py --config configs/default.yaml --device cuda
python plotting/plot_scaling.py \
    --results outputs/efficiency/efficiency_results.json \
    --output-dir outputs/figures
python plotting/plot_results.py --results-dir outputs

echo '✓ Plots → outputs/figures/'

## Cell 12 — Display plots inline

In [ ]:
import matplotlib.pyplot as plt, matplotlib.image as mpimg, os
from pathlib import Path

fig_dir = Path('/kaggle/working/revrag/outputs/figures')
for pdf in sorted(fig_dir.glob('*.pdf')):
    png = pdf.with_suffix('.png')
    os.system(f'convert -density 150 {pdf} {png} 2>/dev/null || true')
    if png.exists():
        print(f'\n--- {pdf.name} ---')
        img = mpimg.imread(str(png))
        plt.figure(figsize=(10, 5)); plt.imshow(img); plt.axis('off')
        plt.tight_layout(); plt.show()

## Cell 13 — Zip everything and download

Download `revrag_results.zip` from the **Output** panel on the right.

In [ ]:
%%bash
cd /kaggle/working
zip -r revrag_results.zip revrag/outputs/ \
    --exclude '*/checkpoints/*'
ls -lh revrag_results.zip
echo '✓ Download revrag_results.zip from the Output panel.'